# BitRoss training (Colab Pro)

This notebook trains the CLIP pixel-DiT + rectified-flow model. The Cursor remote box has **no GPU**, so this is the intended training path.

## GPU (do this first)

**Runtime → Change runtime type**

| GPU | Use it? | Why |
|---|---|---|
| **L4** (default) | **Yes** | Best price:quality. Ada Lovelace, bf16, ~1.7 CU/hr. This 16×16 DiT is tiny; L4 is plenty and much faster than a T4. |
| **T4** | Fine | Cheapest (~1.2 CU/hr). fp16 instead of bf16. Use if L4 is unavailable. |
| **A100 / H100 / G4** | **No** | 3–7× the compute-unit burn, almost no speedup on this model. |

Colab Pro sessions last up to 24h. Checkpoints go to Google Drive so a disconnect is not a wipe. Re-run from the train cell with `RESUME = "auto"`.

## Hugging Face dataset

Gated dump: [OVAWARE/16xModdedMinecraft](https://huggingface.co/datasets/OVAWARE/16xModdedMinecraft) (~1.03M 16×16 RGBA rows: `image`, `file_name`, `type`, `mod_slug`, … — **no captions**).

1. Accept the terms on that page while logged in
2. Add Colab secret `HF_TOKEN`

The prepare cell keeps `type=item` sprites with mixed alpha (drops ≥99% transparent and ≥99% opaque) and writes a Drive cache. Captions are built from `file_name` + `mod_slug`.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime → Change runtime type → GPU → L4 (or T4)."
)
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {name}  CC {cap}  VRAM {vram:.1f} GB  bf16={torch.cuda.is_bf16_supported()}")

expensive = any(k in name.upper() for k in ("A100", "H100", "A800", "PRO 6000", "G4"))
if expensive:
    raise SystemExit(
        f"{name} burns Colab compute units for almost no gain on BitRoss.\n"
        "Runtime → Change runtime type → L4 (preferred) or T4."
    )
if "L4" in name:
    print("L4: best price/quality for this run.")
elif "T4" in name:
    print("T4: cheapest option, slightly slower. Fine for 16x16.")
else:
    print("Unknown GPU — continuing, but L4 is the intended Pro default.")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/BitRoss"
!mkdir -p "{DRIVE_ROOT}/models" "{DRIVE_ROOT}/data"

In [ ]:
# @title Repo + deps
BRANCH = "cursor/hq-cvae-architecture-5a9f"  # @param {type:"string"}
REPO = "https://github.com/OVAWARE/BitRoss.git"

import os
os.chdir("/content")
if not os.path.isdir("/content/BitRoss/.git"):
    !git clone --branch {BRANCH} --depth 1 {REPO}
else:
    os.chdir("/content/BitRoss")
    !git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
os.chdir("/content/BitRoss")
# Colab already ships CUDA torch — do not pip-install the CPU wheel.
%pip install -q -U transformers wandb pillow datasets huggingface_hub

In [ ]:
# @title Prepare Hugging Face dataset (cached on Drive)
FORCE_REBUILD = False  # @param {type:"boolean"}
MAX_SAMPLES = 0  # @param {type:"integer"}

import os
from pathlib import Path

os.chdir("/content/BitRoss")
PROCESSED = "/content/drive/MyDrive/BitRoss/data/processed-items"
os.makedirs(os.path.dirname(PROCESSED), exist_ok=True)

# Token: Colab secret HF_TOKEN, else env.
token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
try:
    from google.colab import userdata
    token = token or userdata.get("HF_TOKEN")
except Exception:
    pass
if not token:
    raise SystemExit(
        "Missing HF_TOKEN. Add it as a Colab secret, and accept the dataset terms at "
        "https://huggingface.co/datasets/OVAWARE/16xModdedMinecraft"
    )
os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token

from huggingface_hub import login
login(token=token)

marker = Path(PROCESSED) / "prepare_stats.json"
if marker.exists() and not FORCE_REBUILD:
    print("Using cached processed dataset:", PROCESSED)
    print(marker.read_text()[:1500])
else:
    extra = []
    if MAX_SAMPLES and MAX_SAMPLES > 0:
        extra += ["--max_samples", str(MAX_SAMPLES)]
    import subprocess, sys
    cmd = [sys.executable, "prepare_dataset.py", "--out_dir", PROCESSED, "--num_proc", "2"] + extra
    print(" ".join(cmd))
    subprocess.check_call(cmd)


In [ ]:
# @title Train
EPOCHS = 800  # @param {type:"integer"}
BATCH_SIZE = 128  # @param {type:"integer"}
RESUME = "auto"  # @param ["auto", ""]
USE_WANDB = False  # @param {type:"boolean"}
WANDB_API_KEY = ""  # @param {type:"string"}

import os, sys, subprocess

os.chdir("/content/BitRoss")
SAVE_DIR = "/content/drive/MyDrive/BitRoss/models"
os.makedirs(SAVE_DIR, exist_ok=True)

if USE_WANDB:
    if WANDB_API_KEY:
        os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    import wandb
    wandb.login()

cmd = [
    sys.executable, "train.py",
    "--processed_dir", "/content/drive/MyDrive/BitRoss/data/processed-items",
    "--save_dir", SAVE_DIR,
    "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", "2",
]
if RESUME:
    cmd += ["--resume", RESUME]
if not USE_WANDB:
    cmd.append("--no_wandb")
print(" ".join(cmd))
subprocess.check_call(cmd)

In [ ]:
# @title Sample from the latest checkpoint
PROMPT = "pixel art minecraft item, diamond sword, blue crystal blade"  # @param {type:"string"}
CFG_SCALE = 2.5  # @param {type:"number"}
STEPS = 20  # @param {type:"integer"}

import os, glob
from IPython.display import display
from PIL import Image

os.chdir("/content/BitRoss")
SAVE_DIR = "/content/drive/MyDrive/BitRoss/models"
ckpts = sorted(glob.glob(SAVE_DIR + "/*.pth"), key=os.path.getmtime)
assert ckpts, f"No checkpoints in {SAVE_DIR}"
ckpt = ckpts[-1]
out = "/content/sample.png"
print("Using", ckpt)
!python generate.py --model_path "{ckpt}" --prompt "{PROMPT}" --output "{out}" --cfg_scale {CFG_SCALE} --steps {STEPS} --size 256
display(Image.open(out))